In [2]:
import sys
!{sys.executable} -m pip install marimo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 104.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.9/269.9 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 872.3/872.3 kB 51.5 MB/s eta 0:00:00
  Attempting uninstall: pyzmq
    Found existing installation: pyzmq 26.2.1
    Uninstalling pyzmq-26.2.1:
      Successfully uninstalled pyzmq-26.2.1


In [1]:
import marimo as mo


ModuleNotFoundError: No module named 'marimo'

In [ ]:
# ============================================================================
#  BRANDED HERO — Pink Elephant palette (magenta #FF2E97 / navy #12203B / cyan #00E5CC)
# ============================================================================
mo.md("""
<div style="background:linear-gradient(135deg,#12203B 0%,#1B2A4A 60%,#2A1B3D 100%);border-radius:16px;padding:28px 32px;border:1.5px solid #FF2E97;">
  <div style="font-size:13px;color:#FF5CAC;letter-spacing:4px;font-weight:700;margin-bottom:8px;">PINK ELEPHANT LIMITED · MIT LICENSE</div>
  <div style="color:#FFFFFF;font-size:34px;font-weight:800;line-height:1.1;">
    THE 48B-S FLAGSHIP — ONE-CLICK DEMO
  </div>
  <div style="color:#00E5CC;font-size:20px;font-weight:700;margin:6px 0 18px;">
    🐘 Sparse Mixture-of-Experts · 47.7B params · 8 experts · Q4_K_M · served by Ollama
  </div>
  <div style="color:#C7D2FE;font-size:14px;line-height:1.7;">
    Just press <b style="color:#FF70B1;">Run all</b>. Cells 1→5 provision everything automatically:
    install Ollama, start the server, download <b>29.3&nbsp;GB</b> of GGUF (once), and register the model
    with the correct <b>ChatML</b> template + conservative sampler. Cell 6 lets you chat live.
  </div>
</div>

<div style="background:#FFF0F7;border-left:4px solid #FF2E97;border-radius:0 10px 10px 0;padding:12px 16px;color:#12203B;font-size:13px;margin-top:12px;">
  <b style="color:#C2185B;">✅ GPU REQUIRED for a snappy demo:</b> click the <b>specs</b> button in the app header
  and attach an <b>NVIDIA RTX PRO 6000 Blackwell</b>, then <b>Run all</b>. Cell 2 auto-detects it.
  <br><b style="color:#C2185B;">ℹ Without a GPU</b> the demo will run CPU-only — correct, but noticeably slower.
  <br><b style="color:#C2185B;">First run downloads 29.3 GB</b> — give it a few minutes. Later runs are instant.
</div>
""")


In [ ]:
import json
import os
import shutil
import subprocess
import time
import urllib.request
from pathlib import Path

# ✓ Config (no hard-coded sandbox path — works anywhere, incl. public Molab cloud)
OLLAMA_MODEL = "pe-48b"
HF_REPO = "pinkelephantlimited/pinkelephant-llm-48b-s-gguf"
GGUF_NAME = "pe_48b_s_dpo_Q4_K_M.gguf"
GGUF_PATH = Path.home() / "models" / GGUF_NAME
OLLAMA_HOST = "http://127.0.0.1:11434"
PATH_PREPEND = "/usr/local/bin:/usr/bin:/bin:/usr/local/cuda/bin"

# ✓ Correct sampler for this MoE (matches the repo Modelfile) — avoids garbled output
DEMO_TEMPERATURE = 0.3
DEMO_REPEAT_PENALTY = 1.0
DEMO_TOP_P = 0.9


def sh(cmd, **kw):
    print(f"$ {cmd}", flush=True)
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)


def ensure_env():
    os.environ["PATH"] = PATH_PREPEND + os.pathsep + os.environ.get("PATH", "")
    os.environ["OLLAMA_MODELS"] = str(Path.home() / ".ollama" / "models")


def ollama_up(timeout=3):
    try:
        with urllib.request.urlopen(OLLAMA_HOST, timeout=timeout):
            return True
    except Exception:
        return False


def have_gpu():
    r = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null")
    if r.returncode == 0 and r.stdout.strip():
        return r.stdout.strip().splitlines()[0]
    return None


In [ ]:
# ============================================================================
#  CELL 1 — PROVISION: environment deps (zstd) + install Ollama (idempotent)
# ============================================================================
def install_ollama():
    ensure_env()
    if shutil.which("ollama"):
        print("✓ Ollama already installed:")
        sh("ollama --version")
        return
    print("Installing zstd (Ollama dependency)…")
    sh("(apt-get update >/dev/null 2>&1 && apt-get install -y zstd >/dev/null 2>&1) || true")
    print("Installing Ollama… (first run only)")
    if r.returncode != 0 or not Path("/tmp/ollama_install.sh").exists():
        raise RuntimeError("Could not download the Ollama installer.")
    sh("nohup bash /tmp/ollama_install.sh > /tmp/ollama_install.log 2>&1 &")
    deadline = time.time() + 600
    while time.time() < deadline:
        if shutil.which("ollama"):
            print("✓ Ollama installed:")
            sh("ollama --version")
            return
        time.sleep(5)
    sh("tail -20 /tmp/ollama_install.log 2>/dev/null")
    raise RuntimeError("Ollama install did not finish in 10 minutes.")


install_ollama()


In [ ]:
# ============================================================================
#  CELL 2 — PROVISION: GPU check + start the Ollama server
# ============================================================================
def start_server():
    ensure_env()
    gpu = have_gpu()
    if gpu:
        print(f"✅ GPU detected: {gpu}")
        print("  → Ollama will offload the full model to GPU for fast generation.")
    else:
        print("⚠️  No NVIDIA GPU detected — running CPU-only (slower but works).")
        print("  💡 For a fast demo, click the header 'specs' button and attach an")
        print("     NVIDIA RTX PRO 6000 Blackwell, then re-run this cell.")
    if ollama_up():
        print(f"✓ Ollama server already running on {OLLAMA_HOST}")
        return
    print("Starting Ollama server…")
    sh("(setsid nohup /usr/local/bin/ollama serve > /tmp/ollama_serve.log 2>&1 &) ; sleep 6")
    deadline = time.time() + 60
    while time.time() < deadline:
        if ollama_up():
            print("✓ Ollama server is up.")
            return
        time.sleep(3)
    sh("tail -15 /tmp/ollama_serve.log 2>/dev/null")
    raise RuntimeError("Ollama server did not start in 60s.")


start_server()


In [ ]:
# ============================================================================
#  CELL 3 - PROVISION: Obtain the 48B-S GGUF (~29.3 GB, Q4_K_M)
#  Investor-safe: 1) reuse an already-downloaded HF cache blob (no re-download),
# ============================================================================
def _sh_live(cmd):
    print(f"$ {cmd}", flush=True)
    return subprocess.run(cmd, shell=True).returncode

def _ensure_real_file(path):
    if path.is_symlink() or (path.exists() and not path.is_file()):
        try:
            path.unlink(); print("removed stale symlink/non-file at", path)
        except OSError as e:
            print("could not remove stale path:", repr(e)[:120])
    path.parent.mkdir(parents=True, exist_ok=True)

def download_model():
    ensure_env()
    EXPECTED_SIZE = 29_255_006_624  # pe_48b_s_dpo_Q4_K_M.gguf (verified via HEAD)
    # Path A: huggingface_hub cache (reuses already-downloaded file, no 29 GB re-fetch)
    try:
        from huggingface_hub import hf_hub_download
        cached = Path(hf_hub_download(repo_id=HF_REPO, filename=GGUF_NAME))
        if cached.exists() and cached.is_file() and cached.stat().st_size >= EXPECTED_SIZE:
            _ensure_real_file(GGUF_PATH)
            try:
                os.link(cached, GGUF_PATH)
            except OSError:
                shutil.copy2(cached, GGUF_PATH)
            print(f"✓ Reused HF cache ({GGUF_PATH.stat().st_size/1e9:.2f} GB)")
            return
    except Exception as e:
        print("(no cached copy; will download via curl):", repr(e)[:100])
    # Path B: raw curl, IPv4-only (avoids unreachable IPv6 on Molab), resumable + verified
    from urllib.parse import quote
    _ensure_real_file(GGUF_PATH)
    url = f"https://huggingface.co/{HF_REPO}/resolve/main/{quote(GGUF_NAME)}"
    if GGUF_PATH.is_file() and GGUF_PATH.stat().st_size >= EXPECTED_SIZE:
        print(f"✓ Downloaded ({GGUF_PATH.stat().st_size/1e9:.2f} GB)")
        return
    print("⬇️  Downloading 29.3 GB GGUF — first run takes ~10-30 min. Progress below updates live.\n")
    for attempt in range(1, 201):
        if GGUF_PATH.is_file() and GGUF_PATH.stat().st_size >= EXPECTED_SIZE:
            print(f"\n✓ Downloaded ({GGUF_PATH.stat().st_size/1e9:.2f} GB)")
            return
        cur = GGUF_PATH.stat().st_size if GGUF_PATH.is_file() else 0
        print(f"[attempt {attempt}] {cur/1e9:.2f} / {EXPECTED_SIZE/1e9:.2f} GB so far\n")
        if rc == 0 and GGUF_PATH.is_file() and GGUF_PATH.stat().st_size >= EXPECTED_SIZE:
            print(f"\n✓ Downloaded ({GGUF_PATH.stat().st_size/1e9:.2f} GB)")
            return
        time.sleep(3)
    raise RuntimeError(f"GGUF download did not complete ({GGUF_PATH.stat().st_size/1e9:.2f} GB)")

download_model()



In [ ]:
# ============================================================================
#  CELL 4 — PROVISION: Register the model with Ollama (CORRECTED ChatML Modelfile)
# ============================================================================
def create_model():
    ensure_env()
    modelfile = (
        f"FROM {GGUF_PATH}\n"
        "# Pink Elephant 48B-S DPO (ChatML) — conservative samplers\n"
        "PARAMETER temperature 0.3\n"
        "PARAMETER repeat_penalty 1.0\n"
        "PARAMETER top_p 0.9\n"
        "PARAMETER num_ctx 4096\n"
        'TEMPLATE """{{- if .System }}<|im_start|>system\n'
        "{{ .System }}<|im_end|>\n"
        "{{- end }}<|im_start|>user\n"
        "{{ .Prompt }}<|im_end|>\n"
        "<|im_start|>assistant\n"
        '"""\n'
    )
    mf = str(Path.home() / "Modelfile.pe-48b")
    Path(mf).write_text(modelfile)
    # Re-create every time so the correct ChatML template is always applied.
    print("Registering model with ChatML template + safe sampler…")
    r = sh(f"ollama create {OLLAMA_MODEL} -f {mf}")
    if r.returncode != 0:
        raise RuntimeError("ollama create failed: " + (r.stderr or r.stdout)[-400:])
    print(f"✓ Model '{OLLAMA_MODEL}' registered:")
    sh("ollama list")


create_model()


In [ ]:
# ============================================================================
#  CELL 5 — PRE-CHECK: Demo readiness + smoke test
# ============================================================================
def readiness():
    ensure_env()
    report = []
    report.append(("✓" if shutil.which("ollama") else "✗", "Ollama installed", "run cell 1"))
    report.append(("✓" if ollama_up() else "✗", "Ollama server", "port 11434" if ollama_up() else "run cell 2"))
    report.append(("✓" if (GGUF_PATH.exists() and GGUF_PATH.stat().st_size > 28e9) else "✗",
                   "48B-S GGUF",
                   f"{GGUF_PATH.stat().st_size/1e9:.1f} GB on disk" if GGUF_PATH.exists() else "run cell 3"))
    gpu = have_gpu()
    report.append(("✓" if gpu else "ℹ", "GPU (recommended)", gpu or "CPU-only fallback — slower"))
    listed = sh("ollama list")
    reg = OLLAMA_MODEL in (listed.stdout or "")
    report.append(("✓" if reg else "✗", "Model registered", OLLAMA_MODEL if reg else "run cell 4"))

    print("=" * 62)
    print("  DEMO READINESS REPORT")
    print("=" * 62)
    for mark, stage, state in report:
        print(f"  {mark}  {stage:<22} {state}")
    print("=" * 62)

    if all(m == "✓" for m, _, _ in report):
        print("\nSmoke test: 'What is the capital of France?' (a real question, not a bare prompt)…")
        payload = {"model": OLLAMA_MODEL,
                   "messages": [{"role": "user", "content": "What is the capital of France?"}],
                   "stream": False,
                   "options": {"num_predict": 16, "temperature": DEMO_TEMPERATURE,
                               "repeat_penalty": DEMO_REPEAT_PENALTY, "top_p": DEMO_TOP_P}}
        req = urllib.request.Request(f"{OLLAMA_HOST}/api/chat", data=json.dumps(payload).encode(),
                                     headers={"Content-Type": "application/json"})
        try:
            t0 = time.time()
            with urllib.request.urlopen(req, timeout=600) as resp:
                d = json.loads(resp.read())
            ans = d.get("message", {}).get("content", "")
            print(f"  ✓ Ready. Round-trip {time.time()-t0:.1f}s.")
            print(f"  Smoke answer: {ans!r}")
        except Exception as e:
            print("  ✗ Smoke test failed:", repr(e)[:200])
    else:
        print("\nComplete the failing stages before demoing (cells above).")


readiness()


In [ ]:
# ============================================================================
#  CELL 6a - DEMO prompt dropdown (created here; .value read in next cell)
# ============================================================================
DEMO_PROMPTS = {
    "Mixture-of-experts in plain English": "Explain what a mixture-of-experts (MoE) language model is, in plain language a non-engineer would understand. Keep it to four sentences.",
    "Write code": "Write a short, clean Python function that checks whether a given integer is prime, with a few example calls.",
    "Explain our efficiency story": "In three sentences, argue why a sparse 48-billion-parameter model with 40 layers and 8 experts can rival dense models that are far larger — and why that matters for the cost of running AI at scale.",
    "Tell a founding story": "Write a 40-word origin story for a small AI lab named after a pink elephant.",
}

prompt_choice = mo.ui.dropdown(
    list(DEMO_PROMPTS.keys()),
    value="Mixture-of-experts in plain English",
    label="Choose a demo prompt",
)
prompt_choice


In [ ]:
# ============================================================================
#  CELL 6b - LIVE: chat with the model (Ollama /api/chat, ChatML in template)
# ============================================================================
PROMPT = DEMO_PROMPTS[prompt_choice.value]


def chat(prompt, temperature=DEMO_TEMPERATURE, max_tokens=420):
    ensure_env()
    payload = {"model": OLLAMA_MODEL,
               "messages": [{"role": "user", "content": prompt}],
               "stream": False,
               "options": {"temperature": temperature, "repeat_penalty": DEMO_REPEAT_PENALTY,
                           "top_p": DEMO_TOP_P, "num_predict": max_tokens}}
    req = urllib.request.Request(f"{OLLAMA_HOST}/api/chat", data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json"})
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=1800) as resp:
            d = json.loads(resp.read())
    except Exception as e:
        print("ERROR:", repr(e)[:300])
        return ""
    dt = time.time() - t0
    msg = d.get("message", {}).get("content", "")
    n = int(d.get("eval_count") or 0)
    speed = (n / dt) if dt > 0 else 0
    import html as _html
    safe = _html.escape(msg).replace("\n", "<br/>")
    mo.md(
        f'''
<div style="background:#FDEFF7;border:1.5px solid #FF2E97;border-radius:12px;padding:14px 18px;font-family:ui-monospace,Menlo,monospace;">
  <div style="color:#12203B;font-size:12px;margin-bottom:10px;border-bottom:1px dashed #FF5CAC;padding-bottom:8px;">
    <b style="color:#C2185B;">pe-48b</b> &nbsp;·&nbsp; latency <b>{dt:.1f}s</b> &nbsp;·&nbsp; {n} tokens &nbsp;·&nbsp; <b style="color:#C2185B;">{speed:.1f} tok/s</b> &nbsp;·&nbsp; temp {temperature}
  </div>
  <div style="color:#2A1B3D;font-size:14px;line-height:1.7;">{safe}</div>
</div>
'''
    )
    return msg


chat(PROMPT)



### Visualização: Avião Pousando em São Paulo
Como sou um assistente de código, posso usar o IPython para exibir um vídeo representativo do YouTube de um pouso no Aeroporto de Congonhas (CGH) ou Guarulhos (GRU).

In [3]:
from IPython.display import YouTubeVideo
# Exemplo de um pouso em Congonhas (um dos mais icônicos de SP)
video_id = 'j6PBOa_Dcrg'
display(YouTubeVideo(video_id, width=800, height=450))